gpt

In [70]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# Set display option to show full text for DESCRIPTION
pd.set_option('display.max_colwidth', None)

# -----------------------------
# Training Phase: Build Similarity Model
# -----------------------------
def train_similarity_model(data_filepath, composite_cols=None, n_components=40):
    """
    Loads preprocessed data, creates a composite text feature, and trains a TF-IDF
    + TruncatedSVD model. Returns the DataFrame, vectorizer, SVD transformer,
    and the cosine similarity matrix.
    """
    # Load data and fill missing with empty strings
    df = pd.read_csv(data_filepath, dtype=str)
    df = df.fillna("")
    
    # If no composite_cols provided, use these defaults:
    if composite_cols is None:
        composite_cols = ["DESCRIPTION", "Attribut1", "Characteristic", "Rating"]
    
    # Use only the columns that exist
    existing_cols = [col for col in composite_cols if col in df.columns]
    # Create composite text by concatenating selected columns (comma separated)
    df['combined_text'] = df[existing_cols].apply(
        lambda row: ", ".join([str(x).strip() for x in row if str(x).strip() != ""]),
        axis=1
    )
    
    # Use the composite text for vectorization
    text_data = df['combined_text']
    
    # TF-IDF Vectorization with tuned parameters
    vectorizer = TfidfVectorizer(
        stop_words='english', 
        ngram_range=(1,2), 
        max_df=0.85, 
        min_df=2, 
        sublinear_tf=True
    )
    tfidf_matrix = vectorizer.fit_transform(text_data)
    
    # Dimensionality reduction using TruncatedSVD (LSA)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    tfidf_reduced = svd.fit_transform(tfidf_matrix)
    
    # Compute cosine similarity on reduced vectors
    cos_sim_matrix = cosine_similarity(tfidf_reduced, tfidf_reduced)
    
    return df, vectorizer, svd, cos_sim_matrix

def get_top_n_similar(part_index, cos_sim_matrix, n=5):
    """
    Given a part index, returns a list of (index, similarity score) tuples for the top n similar parts.
    Excludes the part itself.
    """
    sim_scores = list(enumerate(cos_sim_matrix[part_index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_n = sim_scores[1:n+1]  # Exclude self (first element)
    return top_n


In [71]:

# -----------------------------
# Testing Phase: Recommendation Interface
# -----------------------------
def recommend_for_part(df, cos_sim_matrix, part_identifier, n=5):
    """
    Given a part index (int) or a part ID (str), returns two pandas DataFrames:
      1. The input part's details (ID and DESCRIPTION).
      2. Detailed recommendations including the alternative part's ID, Similarity Score,
         DESCRIPTION, and Combined Text.
    """
    # Determine index from part_identifier
    if isinstance(part_identifier, int):
        idx = part_identifier
    else:
        idx_list = df.index[df['ID'] == part_identifier].tolist()
        if not idx_list:
            print(f"Part ID '{part_identifier}' not found!")
            return None, None
        idx = idx_list[0]
    
    # Get the input part's details (ID and full DESCRIPTION)
    input_part_df = df.loc[[idx], ['ID', 'DESCRIPTION']]
    
    # Get top n similar parts for the specified index
    top_sim = get_top_n_similar(idx, cos_sim_matrix, n=n)
    
    recommendations = []
    for i, score in top_sim:
        recommendations.append({
            "ID": df.iloc[i]['ID'],
            "Similarity Score": np.round(score, 3),
            "DESCRIPTION": df.iloc[i]['DESCRIPTION']
            # ,
            # "Combined Text": df.iloc[i]['combined_text']
        })
    recs_df = pd.DataFrame(recommendations)
    return input_part_df, recs_df

# -----------------------------
# Main Execution: Training & Interactive Testing
# -----------------------------
if __name__ == "__main__":
    # Train the model (adjust file path as needed)
    data_filepath = "Parts_processed.csv"
    df, vectorizer, svd, cos_sim_matrix = train_similarity_model(data_filepath)
    
    # Interactive testing: Manually input a Part ID
    user_input = input("Enter a Part ID for recommendations: ").strip()
    
    input_part_df, detailed_recs = recommend_for_part(df, cos_sim_matrix, user_input, n=5)
    
    if input_part_df is not None and detailed_recs is not None:
        print("\nInput Part Information:")
        display(input_part_df)  # In Jupyter Notebook, display() renders a full DataFrame
        
        print(f"\nDetailed Recommendations for Part ID: {user_input}")
        display(detailed_recs)



Input Part Information:


,ID,DESCRIPTION
0,A1,"Indicator Red Fast Movement 1.6A 250V Holder Plastic 5 X 20mm Ceramic Box CCC/PSE/VDE/cULus Electric Indicator, Very Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm"



Detailed Recommendations for Part ID: A1


,ID,Similarity Score,DESCRIPTION
0,A253,0.864,"Indicator Red Fast Movement 1.6A 250V Holder Plastic 5 X 20mm Ceramic Bulk CCC/CE/CSA/KC/PSE/SEMKO/UL/VDE Electric Indicator, Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm"
1,A254,0.827,"Indicator Red Fast Movement 1.6A 250V Holder Plastic Melf 5 X 20mm Ceramic CCC/CE/CSA/PSE/SEMKO/UL/VDE Electric Indicator, Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder"
2,A91,0.809,"Red Indicator, 5 X 20 mm, Quick-Movement F, L, 250 VAC Electric Indicator, Fast Blow, 1.6A, 250VAC, 35A (IR), Inline/holder, 5x20mm"
3,A252,0.785,"Indicator Red Fast Movement 1.6A 250V Axial 5 X 20mm Ceramic Bulk CCC/CE/CSA/KC/PSE/SEMKO/UL/VDE Electric Indicator, Fast Blow, 1.6A, 250VAC, 1500A (IR), Through Hole, 5x20mm"
4,A458,0.766,"Indicator Red Fast Movement 1.6A 250V Holder Plastic Melf 6.3 X 32mm Glass CSA/UL Electric Indicator, Fast Blow, 1.6A, 250VAC, 100A (IR), Inline/holder"


claude

In [20]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK resources (only needed first time)
# nltk.download('stopwords')
# nltk.download('wordnet')

class SimilarPartsFinder:
    def __init__(self, dataframe):
        self.df = dataframe
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()
        self.tfidf_vectorizer = None
        self.tfidf_matrix = None
        
    def preprocess_text(self, text):
        """Preprocess text by lowercasing, removing punctuation, and lemmatizing"""
        if text == 'NA' or pd.isna(text):
            return ''
            
        # Convert to lowercase
        text = text.lower()
        
        # Remove numbers with units (like 250V, 1.6A) but keep the unit
        text = re.sub(r'(\d+\.?\d*)([a-zA-Z]+)', r' \2 ', text)
        
        # Remove punctuation
        text = re.sub(r'[^\w\s]', ' ', text)
        
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Lemmatize and remove stopwords
        words = text.split()
        words = [self.lemmatizer.lemmatize(word) for word in words if word not in self.stop_words]
        
        return ' '.join(words)
    
    def extract_additional_features(self, row):
        """Extract additional features from other columns"""
        features = []
        
        # Add important technical specifications
        important_columns = ['Attribut1', 'Characteristic', 'Material', 'Mounting', 'Mounting Feature', 
                            'Rating', 'Application', 'Maximum AC Voltage Rating']
        for col in important_columns:
            if col in row and row[col] != 'NA' and not pd.isna(row[col]):
                features.append(str(row[col]).lower())
        
        return ' '.join(features)
    
    def create_enhanced_description(self, row):
        """Create enhanced description combining the original description with key features"""
        processed_desc = self.preprocess_text(row['DESCRIPTION'])
        additional_features = self.extract_additional_features(row)
        
        # Combine with higher weight on the description
        return processed_desc + ' ' + additional_features
    
    def fit_transform(self):
        """Create TF-IDF vectors from the enhanced descriptions"""
        # Create enhanced descriptions
        self.df['enhanced_description'] = self.df.apply(self.create_enhanced_description, axis=1)
        
        # Create TF-IDF vectorizer
        self.tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(self.df['enhanced_description'])
        
        return self.tfidf_matrix
    
    def find_similar_parts(self, part_id, top_n=5):
        """Find top_n similar parts for a given part_id"""
        if part_id not in self.df['ID'].values:
            return f"Part ID {part_id} not found in the dataset."
        
        # Get the index of the part
        idx = self.df.index[self.df['ID'] == part_id].tolist()[0]
        
        # Calculate cosine similarity
        part_vector = self.tfidf_matrix[idx]
        similarities = cosine_similarity(part_vector, self.tfidf_matrix).flatten()
        
        # Set similarity to -1 for the same part to ensure it won't be selected
        similarities[idx] = -1
        
        # Get indices of top similar parts (ensuring the part itself is not included)
        top_indices = similarities.argsort()[::-1][:top_n]
        
        # Create result dataframe
        result = pd.DataFrame({
            'Similar_Part_ID': self.df.iloc[top_indices]['ID'].values,
            'Similarity_Score': similarities[top_indices],
            'Description': self.df.iloc[top_indices]['DESCRIPTION'].values,
            'Rating': self.df.iloc[top_indices]['Rating'].values
        })
        
        return result

# Sample data for demonstration
def load_sample_data():
    df = pd.read_csv("Parts_processed.csv", dtype=str)
    return df

# Load the sample data and train the model
df = load_sample_data()
finder = SimilarPartsFinder(df)
finder.fit_transform()

print("Model trained successfully!")
print(f"Total parts in dataset: {len(df)}")
print("Sample similar parts for A1:")
print(finder.find_similar_parts('A1', top_n=5))

Model trained successfully!
Total parts in dataset: 992
Sample similar parts for A1:
  Similar_Part_ID  Similarity_Score  \
0              A3          0.793201   
1             A92          0.729187   
2              A5          0.723173   
3             A93          0.696017   
4              A6          0.669122   

                                         Description Rating  
0  Indicator Red Fast Movement 8A 250V Holder Pla...     8A  
1  Indicator Red Fast Movement 3.15A 250V Holder ...  3.15A  
2  Indicator Red Fast Movement 12.5A 250V Holder ...  12.5A  
3  Indicator Red Fast Movement 8A 250V Holder Pla...     8A  
4  Indicator Red Fast Movement 12.5A 250V Holder ...  12.5A  


In [21]:
# This code assumes the training code has been executed and the 'finder' object is available

# Test function to get recommendations for a part ID
def get_part_recommendations(part_id, top_n=5):
    """
    Get recommendations for a specific part ID
    
    Parameters:
    part_id (str): The ID of the part to find alternatives for
    top_n (int): Number of recommendations to return
    
    Returns:
    DataFrame or str: Top similar parts with similarity scores
    """
    if part_id not in finder.df['ID'].values:
        available_parts = ', '.join(finder.df['ID'].values)
        return f"Part ID {part_id} not found. Available parts: {available_parts}"
    
    return finder.find_similar_parts(part_id, top_n=top_n)

# Interactive testing
def run_interactive_test():
    """Run interactive testing where the user can enter part IDs"""
    print("\n" + "="*50)
    print("Interactive Parts Similarity Testing")
    print("Enter a part ID to see similar parts or 'quit' to exit")
    print("="*50)
    
    available_parts = ', '.join(finder.df['ID'].values)
    print(f"Available part IDs: {available_parts}")
    
    while True:
        part_id = input("\nEnter part ID (or 'quit' to exit): ")
        
        if part_id.lower() == 'quit':
            print("Exiting testing mode.")
            break
            
        result = get_part_recommendations(part_id)
        
        if isinstance(result, str):
            print(result)
        else:
            print(f"\nTop 5 similar parts for {part_id}:")
            print(result[['Similar_Part_ID', 'Similarity_Score', 'Rating']])
            
            # Additional validation check
            if part_id in result['Similar_Part_ID'].values:
                print(f"WARNING: Part {part_id} is in its own recommendations!")

# Run batch testing on all parts
def run_batch_test():
    """Test the model on all parts in the dataset"""
    print("\n" + "="*50)
    print("Batch Testing - Finding similar parts for all parts")
    print("="*50)
    
    all_passed = True
    
    for part_id in finder.df['ID'].values:
        print(f"\nTesting part: {part_id}")
        result = get_part_recommendations(part_id, top_n=5)
        
        if isinstance(result, str):
            print(f"Error: {result}")
            all_passed = False
            continue
            
        print(f"Found {len(result)} similar parts")
        
        # Verify the part is not in its own recommendations
        if part_id in result['Similar_Part_ID'].values:
            print(f"FAILED: Part {part_id} is in its own recommendations!")
            all_passed = False
        else:
            print("PASSED: Part not in its own recommendations")
    
    if all_passed:
        print("\nAll tests passed successfully!")
    else:
        print("\nSome tests failed. Please check the output above.")

# Execute the tests
print("\nRunning batch test...")
run_batch_test()

print("\nStarting interactive testing...")
run_interactive_test()


Running batch test...

Batch Testing - Finding similar parts for all parts

Testing part: A1
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A2
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A3
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A4
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A5
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A6
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A7
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A8
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A9
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A10
Found 5 similar parts
PASSED: Part not in its own recommendations

Testing part: A11
Found 5 similar parts
PASSED: Part not in its own recommendation

Deep

In [32]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# Load preprocessed data
df = pd.read_csv("Parts_processed.csv", dtype=str)

# 1. Text Embedding Model
text_model = SentenceTransformer('all-MiniLM-L6-v2')
descriptions = df['DESCRIPTION'].tolist()

# 2. Feature Engineering
# ---------------------------------------------------------------
# Textual Features
text_embeddings = text_model.encode(descriptions)

# Term Frequency Features
tfidf = TfidfVectorizer(ngram_range=(1,2), stop_words='english')
tfidf_matrix = tfidf.fit_transform(descriptions)

# Numerical Features (Extract numerical values from strings)
def extract_numerical_value(s):
    """Extract numerical value from strings like '1.6A' or '250V'"""
    if pd.isna(s):
        return 0
    match = re.search(r'\d+\.?\d*', str(s))
    return float(match.group(0)) if match else 0

# Apply to relevant columns
numerical_features = df[['Rating', 'Maximum AC Voltage Rating', 'Maximum DC Voltage Rating']].applymap(extract_numerical_value)
scaler = MinMaxScaler()
scaled_numerical = scaler.fit_transform(numerical_features)

# 3. Similarity Fusion
# ---------------------------------------------------------------
# Weights (adjust based on domain importance)
weights = {'text': 0.4, 'tfidf': 0.2, 'numerical': 0.4}

# Calculate individual similarity matrices
text_sim = cosine_similarity(text_embeddings)
tfidf_sim = cosine_similarity(tfidf_matrix)
num_sim = cosine_similarity(scaled_numerical)

# Combine similarities
combined_sim = (
    weights['text'] * text_sim +
    weights['tfidf'] * tfidf_sim +
    weights['numerical'] * num_sim
)

# 4. Alternative Parts Retrieval
# ---------------------------------------------------------------
def get_alternatives(part_id, n=5):
    idx = df.index[df['ID'] == part_id].tolist()[0]
    sim_scores = list(enumerate(combined_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]  # Exclude self
    return [(df.iloc[i[0]]['ID'], i[1]) for i in sim_scores]

# 5. Generate Recommendations for All Parts
# ---------------------------------------------------------------
recommendations = {}
for part_id in df['ID']:
    recommendations[part_id] = get_alternatives(part_id)

# 6. Save Results
results = []
for part_id, alts in recommendations.items():
    for alt_id, score in alts:
        results.append({
            'Original Part': part_id,
            'Alternative Part': alt_id,
            'Similarity Score': round(score, 4)
        })

results_df

C:\Users\ranga\AppData\Local\Temp\ipykernel_9120\1616011343.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  numerical_features = df[['Rating', 'Maximum AC Voltage Rating', 'Maximum DC Voltage Rating']].applymap(extract_numerical_value)


,Original Part,Alternative Part,Similarity Score
0,A1,A253,0.8940
1,A1,A254,0.8721
2,A1,A3,0.8676
3,A1,A92,0.8239
4,A1,A97,0.8197
...,...,...,...
4955,A998,A998,1.0000
4956,A998,A993,0.9831
4957,A998,A995,0.9831
4958,A998,A985,0.9756


In [33]:
# Sample validation for part A1
a1_alts = results_df[results_df['Original Part'] == 'A1']
print("Alternatives for A1:")
a1_alts.merge(df[['ID', 'DESCRIPTION']], left_on='Alternative Part', right_on='ID')

Alternatives for A1:


,Original Part,Alternative Part,Similarity Score,ID,DESCRIPTION
0,A1,A253,0.8940,A253,Indicator Red Fast Movement 1.6A 250V Holder P...
1,A1,A254,0.8721,A254,Indicator Red Fast Movement 1.6A 250V Holder P...
2,A1,A3,0.8676,A3,Indicator Red Fast Movement 8A 250V Holder Pla...
3,A1,A92,0.8239,A92,Indicator Red Fast Movement 3.15A 250V Holder ...
4,A1,A97,0.8197,A97,Indicator Red Slow Blow Movement 1.6A 250V Hol...
